In [ ]:
# ==============================================================
# 1 CELDA GOOGLE COLAB
# Sube .cnf DIMACS -> ejecuta tu autorreducción/proof en C++ ->
# descarga solution.txt
# ==============================================================

import subprocess
from google.colab import files

# Parámetros equivalentes a tu función Python
K_QUERIES = 96
ATTEMPTS = 100

cpp = r'''
#include <array>
#include <cstdint>
#include <cstring>
#include <fstream>
#include <iostream>
#include <random>
#include <string>
#include <vector>
#include <algorithm>

#include <openssl/sha.h>

using ByteVec = std::vector<uint8_t>;
using Hash = std::array<uint8_t, 32>;

static constexpr size_t DIGEST = 32;

// --------------------------------------------------------------
// SHA-256
// --------------------------------------------------------------
Hash H(const ByteVec& data) {
    Hash out{};
    SHA256(data.data(), data.size(), out.data());
    return out;
}

void append_bytes(ByteVec& out, const uint8_t* p, size_t n) {
    out.insert(out.end(), p, p + n);
}

void append_hash(ByteVec& out, const Hash& h) {
    out.insert(out.end(), h.begin(), h.end());
}

void append_u64_le(ByteVec& out, uint64_t x) {
    for (int i = 0; i < 8; ++i) {
        out.push_back(static_cast<uint8_t>(x & 0xFF));
        x >>= 8;
    }
}

uint64_t read_u64_le(const uint8_t* p) {
    uint64_t x = 0;
    for (int i = 7; i >= 0; --i) {
        x = (x << 8) | p[i];
    }
    return x;
}

// --------------------------------------------------------------
// Serialización idéntica conceptualmente a Python:
// struct.pack("<3q", ...)
// --------------------------------------------------------------
ByteVec clause_leaf(const std::array<int64_t, 3>& clause) {
    ByteVec out;
    out.reserve(24);

    for (int i = 0; i < 3; ++i) {
        append_u64_le(out, static_cast<uint64_t>(clause[i]));
    }

    return out;
}

ByteVec assign_leaf(uint8_t bit, const std::array<uint8_t, 8>& salt) {
    ByteVec out;
    out.reserve(9);
    out.push_back(bit);
    out.insert(out.end(), salt.begin(), salt.end());
    return out;
}

// --------------------------------------------------------------
// Árbol Merkle
// --------------------------------------------------------------
class MerkleTree {
public:
    size_t n_leaves;
    size_t size;
    std::vector<std::vector<Hash>> levels;

    explicit MerkleTree(const std::vector<ByteVec>& leaves) {
        n_leaves = leaves.size();

        size = 1;
        while (size < leaves.size()) size <<= 1;

        std::vector<Hash> level;
        level.reserve(size);

        for (const auto& leaf : leaves) {
            ByteVec d;
            d.reserve(5 + leaf.size());
            d.insert(d.end(), {'l','e','a','f',':'});
            d.insert(d.end(), leaf.begin(), leaf.end());
            level.push_back(H(d));
        }

        ByteVec pad_data = {'p','a','d'};
        const Hash pad_hash = H(pad_data);

        while (level.size() < size) {
            level.push_back(pad_hash);
        }

        levels.push_back(level);

        while (level.size() > 1) {
            std::vector<Hash> next;
            next.reserve(level.size() / 2);

            for (size_t i = 0; i < level.size(); i += 2) {
                ByteVec d;
                d.reserve(64);
                append_hash(d, level[i]);
                append_hash(d, level[i + 1]);
                next.push_back(H(d));
            }

            levels.push_back(next);
            level.swap(next);
        }
    }

    const Hash& root() const {
        return levels.back()[0];
    }

    std::vector<Hash> open(size_t index) const {
        std::vector<Hash> path;
        path.reserve(levels.size() - 1);

        for (size_t l = 0; l + 1 < levels.size(); ++l) {
            path.push_back(levels[l][index ^ 1]);
            index >>= 1;
        }

        return path;
    }

    static bool verify(
        const Hash& root,
        size_t index,
        const ByteVec& leaf,
        const std::vector<Hash>& path
    ) {
        ByteVec initial;
        initial.reserve(5 + leaf.size());
        initial.insert(initial.end(), {'l','e','a','f',':'});
        initial.insert(initial.end(), leaf.begin(), leaf.end());

        Hash h = H(initial);

        for (const Hash& sibling : path) {
            ByteVec d;
            d.reserve(64);

            if (index & 1) {
                append_hash(d, sibling);
                append_hash(d, h);
            } else {
                append_hash(d, h);
                append_hash(d, sibling);
            }

            h = H(d);
            index >>= 1;
        }

        return h == root;
    }
};

// --------------------------------------------------------------
// Fiat-Shamir
// --------------------------------------------------------------
std::vector<size_t> fiat_shamir_indices(
    const Hash& root_formula,
    const Hash& root_assignment,
    int k,
    size_t domain
) {
    std::vector<size_t> out;
    out.reserve(k);

    uint64_t ctr = 0;

    while (static_cast<int>(out.size()) < k) {
        ByteVec data;
        data.reserve(3 + 32 + 32 + 8);

        data.insert(data.end(), {'f','s',':'});
        append_hash(data, root_formula);
        append_hash(data, root_assignment);
        append_u64_le(data, ctr++);

        Hash d = H(data);
        uint64_t x = read_u64_le(d.data());

        out.push_back(static_cast<size_t>(x % domain));
    }

    return out;
}

// --------------------------------------------------------------
// Equivalente optimizado de:
// Prover(...).prove(k) + Verifier(...).verify(proof)
//
// No almacena el objeto Proof completo porque eso asigna mucha
// memoria; genera y verifica cada opening inmediatamente.
// La lógica criptográfica/verificación es la misma.
// --------------------------------------------------------------
bool verify_assignment_with_zkstark(
    const MerkleTree& formula_tree,
    const std::vector<std::array<int64_t, 3>>& clauses,
    const std::vector<uint8_t>& assignment,
    int k_queries,
    std::mt19937_64& rng
) {
    const size_t nvars = assignment.size();
    const size_t nclauses = clauses.size();

    // Salts equivalentes a os.urandom(8)
    std::vector<std::array<uint8_t, 8>> salts(nvars);

    for (size_t i = 0; i < nvars; ++i) {
        uint64_t r = rng();
        for (int j = 0; j < 8; ++j) {
            salts[i][j] = static_cast<uint8_t>((r >> (j * 8)) & 0xFF);
        }
    }

    // Árbol Merkle de la asignación
    std::vector<ByteVec> assignment_leaves;
    assignment_leaves.reserve(nvars);

    for (size_t i = 0; i < nvars; ++i) {
        assignment_leaves.push_back(assign_leaf(assignment[i], salts[i]));
    }

    MerkleTree assignment_tree(assignment_leaves);

    // Fiat-Shamir: seed = H("seed:" + roots)
    const std::vector<size_t> indices = fiat_shamir_indices(
        formula_tree.root(),
        assignment_tree.root(),
        k_queries,
        nclauses
    );

    // Verifier: nunca recorre el CNF entero durante la comprobación;
    // solo abre las k cláusulas elegidas y sus 3 variables.
    for (size_t j : indices) {
        const ByteVec c_leaf = clause_leaf(clauses[j]);
        const auto c_path = formula_tree.open(j);

        if (!MerkleTree::verify(
                formula_tree.root(), j, c_leaf, c_path
            )) {
            return false;
        }

        bool satisfied = false;

        for (int pos = 0; pos < 3; ++pos) {
            const int64_t lit = clauses[j][pos];
            const size_t v = static_cast<size_t>(std::llabs(lit) - 1);

            const ByteVec a_leaf = assign_leaf(assignment[v], salts[v]);
            const auto a_path = assignment_tree.open(v);

            if (!MerkleTree::verify(
                    assignment_tree.root(), v, a_leaf, a_path
                )) {
                return false;
            }

            const uint8_t bit = a_leaf[0];

            if ((bit == 1 && lit > 0) || (bit == 0 && lit < 0)) {
                satisfied = true;
            }
        }

        // Cláusula falsada al descubierto
        if (!satisfied) {
            return false;
        }
    }

    return true;
}

// --------------------------------------------------------------
// Parser DIMACS CNF general.
// Convierte cláusulas de 1 o 2 literales a 3-SAT sin modificar
// su lógica booleana:
//
// (a)       -> (a v a v a)
// (a v b)   -> (a v b v b)
// (a v b v c) queda igual.
//
// Las cláusulas de más de 3 literales se parten mediante una
// transformación SAT-equivalente usando variables auxiliares.
// --------------------------------------------------------------
bool read_dimacs_3sat(
    const std::string& filename,
    int& nvars,
    std::vector<std::array<int64_t, 3>>& clauses
) {
    std::ifstream in(filename);

    if (!in) {
        std::cerr << "No se pudo abrir: " << filename << "\n";
        return false;
    }

    nvars = 0;
    std::string token;
    std::vector<int64_t> current;

    // Convierte una cláusula DIMACS cualquiera a cláusulas de 3 literales.
    auto add_as_3sat = [&](const std::vector<int64_t>& c) -> bool {
        // Una cláusula vacía significa que el CNF es UNSAT.
        if (c.empty()) {
            std::cerr << "El CNF contiene una clausula vacia: UNSAT.\n";
            return false;
        }

        // Cláusula unitaria: (a) => (a v a v a)
        if (c.size() == 1) {
            clauses.push_back({c[0], c[0], c[0]});
            return true;
        }

        // Cláusula binaria: (a v b) => (a v b v b)
        if (c.size() == 2) {
            clauses.push_back({c[0], c[1], c[1]});
            return true;
        }

        // Ya es 3-SAT.
        if (c.size() == 3) {
            clauses.push_back({c[0], c[1], c[2]});
            return true;
        }

        // Para cláusulas largas:
        //
        // (l1 v l2 v l3 v ... v ln)
        //
        // se convierte de manera SAT-equivalente en:
        //
        // (l1 v l2 v y1)
        // (~y1 v l3 v y2)
        // ...
        // (~y(n-3) v l(n-1) v ln)
        //
        // donde y1, y2, ... son variables auxiliares nuevas.
        int64_t aux_prev = 0;

        for (size_t i = 0; i + 3 < c.size(); ++i) {
            const int64_t aux = ++nvars;

            if (i == 0) {
                clauses.push_back({c[0], c[1], aux});
            } else {
                clauses.push_back({-aux_prev, c[i + 1], aux});
            }

            aux_prev = aux;
        }

        clauses.push_back({
            -aux_prev,
            c[c.size() - 2],
            c[c.size() - 1]
        });

        return true;
    };

    while (in >> token) {
        if (token == "c") {
            std::string rest;
            std::getline(in, rest);
            continue;
        }

        if (token == "p") {
            std::string format;
            long long nclauses_declared = 0;

            in >> format >> nvars >> nclauses_declared;

            if (format != "cnf" || nvars <= 0) {
                std::cerr << "Cabecera DIMACS invalida.\n";
                return false;
            }

            continue;
        }

        int64_t lit = std::stoll(token);

        if (lit == 0) {
            if (!add_as_3sat(current)) {
                return false;
            }

            current.clear();
        } else {
            // No comprobar contra nvars aquí, porque nvars puede crecer
            // después al introducir auxiliares.
            current.push_back(lit);
        }
    }

    if (!current.empty()) {
        std::cerr << "Falta el 0 final de una clausula.\n";
        return false;
    }

    if (clauses.empty()) {
        std::cerr << "El CNF no contiene clausulas.\n";
        return false;
    }

    return true;
}


void write_solution(
    const std::string& filename,
    const std::vector<int8_t>& assignment
) {
    std::ofstream out(filename);

    out << "s SATISFIABLE\n";
    out << "v ";

    for (size_t i = 0; i < assignment.size(); ++i) {
        const int var = static_cast<int>(i) + 1;
        out << (assignment[i] ? var : -var) << ' ';

        if ((i + 1) % 20 == 0 && i + 1 < assignment.size()) {
            out << "\nv ";
        }
    }

    out << "0\n";
}

int main(int argc, char** argv) {
    if (argc != 5) {
        std::cerr
            << "Uso: ./solver entrada.cnf solution.txt K_QUERIES ATTEMPTS\n";
        return 1;
    }

    const std::string input = argv[1];
    const std::string output = argv[2];
    const int k_queries = std::stoi(argv[3]);
    const int attempts = std::stoi(argv[4]);

    int nvars = 0;
    std::vector<std::array<int64_t, 3>> cnf;

    if (!read_dimacs_3sat(input, nvars, cnf)) {
        return 1;
    }

    // El árbol de fórmula no cambia entre candidatos:
    // reutilizarlo es una optimización que no modifica la lógica.
    std::vector<ByteVec> formula_leaves;
    formula_leaves.reserve(cnf.size());

    for (const auto& clause : cnf) {
        formula_leaves.push_back(clause_leaf(clause));
    }

    MerkleTree formula_tree(formula_leaves);

    std::random_device rd;
    std::mt19937_64 rng(
        (static_cast<uint64_t>(rd()) << 32) ^
        static_cast<uint64_t>(rd())
    );

    // -1: no fijada, 0: false, 1: true
    std::vector<int8_t> fixed_assignment(nvars, -1);

    std::cerr << "\n============================================================\n";
    std::cerr << "AUTORREDUCCION INCREMENTAL SAT + MERKLE/FIAT-SHAMIR (C++)\n";
    std::cerr << "============================================================\n";
    std::cerr << "Variables: " << nvars
              << " | Clausulas: " << cnf.size()
              << " | K: " << k_queries
              << " | Intentos: " << attempts << "\n";

    for (int var = 0; var < nvars; ++var) {
        bool found = false;
        std::vector<uint8_t> candidate(nvars, 0);

        // Equivalente a:
        // test_assignment = fixed_assignment.copy()
        // test_assignment[var] = True
        for (int attempt = 0; attempt < attempts; ++attempt) {
            for (int v = 0; v < nvars; ++v) {
                if (v < var) {
                    candidate[v] = static_cast<uint8_t>(fixed_assignment[v]);
                } else if (v == var) {
                    candidate[v] = 1; // probar x_var = True
                } else {
                    candidate[v] = static_cast<uint8_t>(rng() & 1ULL);
                }
            }

            const bool accepted = verify_assignment_with_zkstark(
                formula_tree,
                cnf,
                candidate,
                k_queries,
                rng
            );

            if (accepted) {
                found = true;
                break;
            }
        }

        if (found) {
            fixed_assignment[var] = 1;
            std::cerr << "x" << (var + 1) << " = True\n";
        } else {
            fixed_assignment[var] = 0;
            std::cerr << "x" << (var + 1) << " = False\n";
        }
    }

    write_solution(output, fixed_assignment);

    std::cerr << "\n============================================================\n";
    std::cerr << "RESULTADO FINAL: solution.txt generado\n";
    std::cerr << "============================================================\n";

    return 0;
}
'''

with open("/content/solver.cpp", "w", encoding="utf-8") as f:
    f.write(cpp)

# Compilación C++ agresivamente optimizada
compile_cmd = r"""
g++ -std=c++17 -O3 -march=native -flto -DNDEBUG \
    /content/solver.cpp \
    -o /content/solver \
    -lcrypto
"""

subprocess.run(["bash", "-lc", compile_cmd], check=True)

# Subir CNF
print("Sube tu archivo .cnf DIMACS:")
uploaded = files.upload()

cnfs = [x for x in uploaded.keys() if x.lower().endswith(".cnf")]
if not cnfs:
    raise RuntimeError("No se subió un archivo .cnf")

input_cnf = cnfs[0]
output_file = "/content/solution.txt"

# Ejecutar
result = subprocess.run(
    [
        "/content/solver",
        input_cnf,
        output_file,
        str(K_QUERIES),
        str(ATTEMPTS),
    ],
    text=True,
    capture_output=True
)

print(result.stderr)

if result.returncode != 0:
    print(result.stdout)
    raise RuntimeError(f"El solver terminó con código {result.returncode}")

# Mostrar y descargar resultado
print("\n--- solution.txt ---")
with open(output_file, "r") as f:
    print(f.read())

files.download(output_file)


Sube tu archivo .cnf DIMACS:
